# Comparativa de Agencias de Transporte
Unifica las categorías de todas las agencias y encuentra la más económica por producto.

In [1]:
import openpyxl
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

wb_libro = openpyxl.load_workbook('Libro1.xlsx')
ws_libro = wb_libro['Hoja1']

wb_ag = openpyxl.load_workbook('COMPARATIVA AGENCIAS.xlsx', data_only=True)
print('Hojas disponibles:', wb_ag.sheetnames[:5], '...')

Hojas disponibles: ['Costo ag. Mayo 2026', 'Costo ag. Abril 2026', 'Costo ag. Marzo 2026', 'Costo ag. Febrero 2026', 'Costo.ag Enero 2026'] ...


In [2]:
# ── Cargar Libro1 ──────────────────────────────────────────────────────────
rows = list(ws_libro.iter_rows(values_only=True))
cols = rows[0]
df = pd.DataFrame(rows[1:], columns=cols)
df.columns = [str(c).strip() if c else c for c in df.columns]

# Normalizar columnas clave
df['Familia'] = df['Familia'].fillna('').astype(str).str.strip()
df['Descripción de Producto'] = df['Descripción de Producto'].fillna('').astype(str).str.strip()
df['Ramo'] = df.iloc[:, 20].fillna('').astype(str).str.strip()  # col 21 (0-indexed 20)

print(f'Productos cargados: {len(df)}')
print('Familias:', sorted(df['Familia'].unique()))

Productos cargados: 1437
Familias: ['ACCES. OTRAS M', 'AGR DEL. OTRAS', 'AGR TRAS. OTRAS', 'AGRICOLA DEL.', 'AGRICOLA TRAS.', 'BATERIAS', 'CAM MOTO OTRAS', 'CAMIONETA OTRAS', 'GIGANTE', 'GIGANTE OTRAS M', 'IND. Y VIALES', 'LUBRICANTES', 'MOTO', 'MOTO OTRAS M.', 'PASEO', 'PASEO OTRAS MAR', 'PICK UP', 'VIALES OTRAS M']


In [3]:
# ── Parsear tabla de precios de la hoja más reciente ──────────────────────
SHEET = 'Costo ag. Mayo 2026'  # ← cambiar para otro mes
ws_ag = wb_ag[SHEET]
ag_rows = list(ws_ag.iter_rows(values_only=True))

# Fila 6 (idx=5) = nombres de agencias; fila 7 (idx=6) = headers; fila 8+ = datos
agency_name_row = ag_rows[5]
header_row      = ag_rows[6]
data_rows       = ag_rows[7:]

# Detectar columnas por agencia (agrupadas: nombre en pos par, precio en pos impar)
ncols = len(agency_name_row)
agency_col_groups = {}   # agency_name -> [col_indices]
col = 0
while col < ncols:
    name = agency_name_row[col]
    if name and str(name).strip():
        next_agency = col + 1
        while next_agency < ncols and (agency_name_row[next_agency] is None or str(agency_name_row[next_agency]).strip() == ''):
            next_agency += 1
        agency_col_groups[str(name).strip()] = list(range(col, next_agency))
        col = next_agency
    else:
        col += 1

print('Agencias detectadas:')
for ag, cols_idx in agency_col_groups.items():
    hdrs = [header_row[c] for c in cols_idx if header_row[c]]
    print(f'  {ag}: cols {cols_idx} → {hdrs}')

Agencias detectadas:
  DAC: cols [0, 1] → ['Categorias', 'Precios iva incl']
  NASAZZI: cols [2, 3] → ['Categorias', 'Precios iva incl']
  MEGAM: cols [4, 5] → ['Categorias', 'Precios iva incl']
  Transportes Nagar sas (Expreso Ruta 1): cols [6, 7] → ['Categorias', 'Precios iva incl.  ']
  SELEGUIN: cols [8, 9] → ['Categorias', 'Precios iva incl. Actualizados']
  EXPRESO ROCHA: cols [10, 11] → ['Categorias', 'Precios iva incl.']
  BULEVAR (ACC): cols [12, 13] → ['Categorias', 'Precios iva incl.']
  PERICO: cols [14, 15] → ['Categorias', 'Precios iva incl.']
  TRUJILLO: cols [16, 17] → ['Categorias', 'Precios iva incl.']
  GONFER: cols [18, 19, 20, 21] → ['Categorias', 'Precios sin IVA (Empresa literal E)', '% aumento', 'Precio Actualizado sin IVA (Empresa literal E)']
  3EME (El Chambon): cols [22, 23] → ['Categorias', 'Precios iva incl.']
  Martin Escudero (Lascano): cols [24, 25] → ['Categorias ', 'total iva incl.']
  Arzuaga: cols [26, 27, 28] → ['Categorias ', ' sin levante  IVA in

In [4]:
# ── Extraer precios + CANJE + FRECUENCIA por agencia ──────────────────────
canje_row = ag_rows[2]   # fila 3 del xlsx
freq_row  = ag_rows[4]   # fila 5 del xlsx

def pick_price_col(agency_name, cols_in_group):
    cat_col   = cols_in_group[0]
    price_col = cols_in_group[1] if len(cols_in_group) > 1 else cols_in_group[0]
    if 'GONFER' in agency_name.upper() and len(cols_in_group) >= 4:
        price_col = cols_in_group[3]
    return cat_col, price_col

def parse_canje(val):
    if isinstance(val, (int, float)):
        return min(val / 100.0 if val >= 1 else val, 0.99)
    return 0.0

def parse_frequency(text):
    """Devuelve (etiqueta_corta, dias_por_semana).
    Sin especificar → L a V (5 días) según criterio del usuario."""
    if not text or str(text).strip() == '':
        return 'L a V', 5
    t = str(text).lower()
    if 'cuando tiene carga' in t:
        return 'Bajo pedido', 1        # irregular, solo cuando junta carga
    if 'mart y jue' in t or 'martes y jueves' in t:
        return 'Mar y Jue', 2
    if 'l, mier y v' in t or 'l,mier' in t:
        return 'L, Mier y V', 3
    if 'l a v' in t or 'lunes a viernes' in t:
        return 'L a V', 5
    # Si dice algo pero no reconocido → default 5 días
    raw = str(text).replace('Frecuencia:', '').strip()
    return raw, 5

agency_prices  = {}
agency_meta    = {}
agency_canje   = {}
agency_freq    = {}   # (etiqueta, dias_semana)

for ag, cols_idx in agency_col_groups.items():
    cat_col, price_col = pick_price_col(ag, cols_idx)

    # Precios
    prices = {}
    for row in data_rows:
        cat   = row[cat_col]   if cat_col   < len(row) else None
        price = row[price_col] if price_col < len(row) else None
        if cat and isinstance(cat, str) and cat.strip() and price and isinstance(price, (int, float)):
            prices[cat.strip()] = float(price)
    agency_prices[ag] = prices
    agency_meta[ag]   = 'sin IVA' if 'GONFER' in ag.upper() else 'IVA incl.'

    # Canje
    canje_col = cols_idx[2] if ('GONFER' in ag.upper() and len(cols_idx) >= 3) else price_col
    raw_canje = canje_row[canje_col] if canje_col < len(canje_row) else None
    agency_canje[ag] = parse_canje(raw_canje)

    # Frecuencia (siempre está en la columna de categoría del grupo)
    raw_freq = freq_row[cat_col] if cat_col < len(freq_row) else None
    agency_freq[ag] = parse_frequency(raw_freq)

print(f'{"Agencia":<45} {"Cats":>5}  {"IVA":>12}  {"Canje":>6}  {"Frecuencia":<14} {"Días/sem"}')
print('─' * 100)
for ag in agency_prices:
    freq_label, freq_days = agency_freq[ag]
    c = agency_canje[ag]
    print(f'{ag:<45} {len(agency_prices[ag]):>5}  {agency_meta[ag]:>12}  {c*100:>5.0f}%  {freq_label:<14} {freq_days}')

Agencia                                        Cats           IVA   Canje  Frecuencia     Días/sem
────────────────────────────────────────────────────────────────────────────────────────────────────
DAC                                               9     IVA incl.     10%  L a V          5
NASAZZI                                          20     IVA incl.     40%  L a V          5
MEGAM                                             9     IVA incl.     25%  L a V          5
Transportes Nagar sas (Expreso Ruta 1)            9     IVA incl.      0%  L a V          5
SELEGUIN                                         12     IVA incl.      1%  L a V          5
EXPRESO ROCHA                                     8     IVA incl.      0%  L, Mier y V    3
BULEVAR (ACC)                                    22     IVA incl.     25%  L a V          5
PERICO                                           18     IVA incl.      0%  Mar y Jue      2
TRUJILLO                                         11     IVA incl

In [5]:
# ── CATEGORÍAS UNIFICADAS y MAPEO AGENCIA → CATEGORÍA ───────────────────
# Cada agencia usa nombres propios. Este dict traduce cada categoría unificada
# al label exacto de la tabla de precios de cada agencia.
# None = no cotiza ese tipo de producto.
# EDITAR si cambian los nombres.

UNIFIED_CATS = {
    'cub_moto':          'Cubierta Moto',
    'cub_auto_r12_r14':  'Cubierta Auto R12-R14',
    'cub_auto_r15_r19':  'Cubierta Auto/Camioneta R15-R19',
    'cub_camion_r20_r22':'Cubierta Camión R20-R22.5',
    'cub_agro_del':      'Cubierta Agrícola Delantera',
    'cub_agro_tras_med': 'Cubierta Agrícola Trasera Mediana 24-26"',
    'cub_agro_tras_gde': 'Cubierta Agrícola Trasera Grande 28-36"',
    'cub_agro_tras_xgde':'Cubierta Agrícola/Vial Extra Grande 38"+',
    'camara':            'Cámara (cualquier tamaño)',
    'bateria_chica':     'Batería Chica (≤110 Ah)',
    'bateria_grande':    'Batería Grande (>110 Ah)',
    'lubricante_caja':   'Lubricante en caja/lata',
    'lubricante_tambor': 'Lubricante Tambor 205L',
    'bulto_general':     'Bulto General',
}

MAPPING = {
    'DAC': {
        'cub_moto':          'Atado moto',
        'cub_auto_r12_r14':  'Rodado 12,13,14',
        'cub_auto_r15_r19':  'Rodado 15 a 18',
        'cub_camion_r20_r22':'Rodado 20 a 22,5',
        'cub_agro_del':      'Cubierta tractor AGRO 24 a 42',
        'cub_agro_tras_med': 'Cubierta tractor AGRO 24 a 42',
        'cub_agro_tras_gde': 'Cubierta tractor AGRO 24 a 42',
        'cub_agro_tras_xgde':'Cubierta tractor AGRO 24 a 42',
        'camara':            'Bulto hasta 15kg',
        'bateria_chica':     'Bateria chica',
        'bateria_grande':    'Bateria grande',
        'lubricante_caja':   'Bulto hasta 15kg',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bulto hasta 15kg',
    },
    'NASAZZI': {
        'cub_moto':          'Rodado 15 a 19',
        'cub_auto_r12_r14':  'Rodado 13 y 14',
        'cub_auto_r15_r19':  'Rodado 15 a 19',
        'cub_camion_r20_r22':'Rodado 20 a 22,5',
        'cub_agro_del':      'Rodado 24 y 25',
        'cub_agro_tras_med': 'Rodado 24 y 25',
        'cub_agro_tras_gde': 'Rodado 26 a 32',
        'cub_agro_tras_xgde':'Cub viales de 38 a 46 extra grande',
        'camara':            'Bultos',
        'bateria_chica':     'Bat hasta 110 amp',
        'bateria_grande':    'Bat mayor 110 amp',
        'lubricante_caja':   'Bultos lubricantes',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bultos',
    },
    'MEGAM': {
        'cub_moto':          'Bat y bultos',
        'cub_auto_r12_r14':  'Cub 13 y 14',
        'cub_auto_r15_r19':  'Cubierta auto (15 a 17)',
        'cub_camion_r20_r22':'Cubierta camion',
        'cub_agro_del':      'Tractor chico',
        'cub_agro_tras_med': 'Tractor chico',
        'cub_agro_tras_gde': 'Tractor grande',
        'cub_agro_tras_xgde':'Tractor grande',
        'camara':            'Bat y bultos',
        'bateria_chica':     'Bat y bultos',
        'bateria_grande':    'Bat y bultos',
        'lubricante_caja':   'Bat y bultos',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bat y bultos',
    },
    'Transportes Nagar sas (Expreso Ruta 1)': {
        'cub_moto':          'Auto',
        'cub_auto_r12_r14':  'Auto',
        'cub_auto_r15_r19':  'Camioneta',
        'cub_camion_r20_r22':'Camion',
        'cub_agro_del':      'Tractor del y 17,5',
        'cub_agro_tras_med': 'Tractor traseras',
        'cub_agro_tras_gde': 'Tractor grande',
        'cub_agro_tras_xgde':'Tractor grande',
        'camara':            'Bultos',
        'bateria_chica':     'Baterias',
        'bateria_grande':    'Baterias',
        'lubricante_caja':   'Bultos',
        'lubricante_tambor': 'Tambor',
        'bulto_general':     'Bultos',
    },
    'SELEGUIN': {
        'cub_moto':          'Bultos, atados, bolsas, cajas',
        'cub_auto_r12_r14':  'Cubierta auto',
        'cub_auto_r15_r19':  'Cubierta camioneta',
        'cub_camion_r20_r22':'Cubierta camion',
        'cub_agro_del':      'Bultos, atados, bolsas, cajas',
        'cub_agro_tras_med': 'Tractor mediana 24 a 26',
        'cub_agro_tras_gde': 'Tractor grande 28 a 30',
        'cub_agro_tras_xgde':'Tractor extra grande +30',
        'camara':            'Bultos, atados, bolsas, cajas',
        'bateria_chica':     'Baterias chicas',
        'bateria_grande':    'Baterias grandes',
        'lubricante_caja':   'Bultos, atados, bolsas, cajas',
        'lubricante_tambor': None,
        'bulto_general':     'Bultos, atados, bolsas, cajas',
    },
    'EXPRESO ROCHA': {
        'cub_moto':          'Bultos camaras',
        'cub_auto_r12_r14':  'Cubierta chica hasta 205/70.16',
        'cub_auto_r15_r19':  'Cubierta grande hasta 750.16',
        'cub_camion_r20_r22':'Cubierta camion',
        'cub_agro_del':      'Neumatico agricolas y viales hasta 18.4/15-26',
        'cub_agro_tras_med': 'Neum. Agricolas y viales mas de 23.1/18-26',
        'cub_agro_tras_gde': 'Neum. Agricolas y viales mas de 23.1/18-26',
        'cub_agro_tras_xgde':'Neum. Agricolas y viales mas de 23.1/18-26',
        'camara':            'Bultos camaras',
        'bateria_chica':     'Baterias',
        'bateria_grande':    'Baterias',
        'lubricante_caja':   'Bultos camaras',
        'lubricante_tambor': None,
        'bulto_general':     'Bultos camaras',
    },
    'BULEVAR (ACC)': {
        'cub_moto':          'Atado cub moto',
        'cub_auto_r12_r14':  'Cub auto  hasta rod 14',
        'cub_auto_r15_r19':  'Cub camioneta 15 a 19',
        'cub_camion_r20_r22':'Cub camion',
        'cub_agro_del':      'Cub tractor chica (hasta 22)',
        'cub_agro_tras_med': 'Cub tractor mediana (24)',
        'cub_agro_tras_gde': 'Cub tractor grande (de 25 a 36)',
        'cub_agro_tras_xgde':'Cub vial rod 38 a 42',
        'camara':            'Bolsas',
        'bateria_chica':     'Bat chicas',
        'bateria_grande':    'Bat grandes',
        'lubricante_caja':   'Caja aceite 1 y 4lts',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bolsas',
    },
    'PERICO': {
        'cub_moto':          'Atado cub moto',
        'cub_auto_r12_r14':  'Cub auto  hasta rod 14',
        'cub_auto_r15_r19':  'Cub camioneta 15 a 19',
        'cub_camion_r20_r22':'Cub camion',
        'cub_agro_del':      'Cub tractor chica',
        'cub_agro_tras_med': 'Cub tractor mediana',
        'cub_agro_tras_gde': 'Cub tractor grande',
        'cub_agro_tras_xgde':'Cub vial rod 38 a 42',
        'camara':            'Bolsas',
        'bateria_chica':     'Bat chicas',
        'bateria_grande':    'Bat grandes',
        'lubricante_caja':   'Caja aceite 1 y 4lts',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bolsas',
    },
    'TRUJILLO': {
        'cub_moto':          'Bultos',
        'cub_auto_r12_r14':  'Paseo hasta 16',
        'cub_auto_r15_r19':  'Paseo hasta 16',
        'cub_camion_r20_r22':'Camion',
        'cub_agro_del':      'Agricola delantera',
        'cub_agro_tras_med': 'Agricola mediana',
        'cub_agro_tras_gde': 'Agricola extra grande',
        'cub_agro_tras_xgde':'Agricola extra grande',
        'camara':            'Bultos',
        'bateria_chica':     'Bat hasta 150 amp',
        'bateria_grande':    'Bat + 150 amp',
        'lubricante_caja':   'Cajas lubricantes',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bultos',
    },
    'GONFER': {
        'cub_moto':          'Bultos',
        'cub_auto_r12_r14':  'Cubierta de auto',
        'cub_auto_r15_r19':  'Cubierta de camioneta',
        'cub_camion_r20_r22':'Cubierta de camion chico',
        'cub_agro_del':      'Agricola chica',
        'cub_agro_tras_med': 'Agricola hasta rod 28',
        'cub_agro_tras_gde': 'Agricola grande rod 30 en adelante',
        'cub_agro_tras_xgde':'Agricola grande rod 30 en adelante',
        'camara':            'Bultos',
        'bateria_chica':     'Bat hasta 110 amp',
        'bateria_grande':    'Bat mayor 110 amp',
        'lubricante_caja':   'Bultos lubricantes',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bultos',
    },
    '3EME (El Chambon)': {
        'cub_moto':          'Atado cub moto',
        'cub_auto_r12_r14':  'Cub auto  hasta rod 14',
        'cub_auto_r15_r19':  'Cub camioneta 15 a 19',
        'cub_camion_r20_r22':'Cub camion',
        'cub_agro_del':      'Cub tractor chica',
        'cub_agro_tras_med': 'Cub tractor mediana',
        'cub_agro_tras_gde': 'Cub tractor grande',
        'cub_agro_tras_xgde':'Cub vial rod 38 a 42',
        'camara':            'Bolsas',
        'bateria_chica':     'Bat chicas',
        'bateria_grande':    'Bat grandes',
        'lubricante_caja':   'Caja aceite 1 y 4lts',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bolsas',
    },
    'Martin Escudero (Lascano)': {
        'cub_moto':          'Auto',
        'cub_auto_r12_r14':  'Auto',
        'cub_auto_r15_r19':  'Camioneta',
        'cub_camion_r20_r22':'Camion',
        'cub_agro_del':      'Agricola chica',
        'cub_agro_tras_med': 'Agricola mediana',
        'cub_agro_tras_gde': 'Agricola grande',
        'cub_agro_tras_xgde':'Agricola grande',
        'camara':            'Bultos/atados',
        'bateria_chica':     'Baterias chicas',
        'bateria_grande':    'Baterias grandes',
        'lubricante_caja':   'Bultos lubricantes',
        'lubricante_tambor': 'Tambor aceite',
        'bulto_general':     'Bultos/atados',
    },
    'Arzuaga': {
        'cub_moto':          'Atado cubiertas moto',
        'cub_auto_r12_r14':  'cubiertas r13 r14',
        'cub_auto_r15_r19':  'cubiertas r16 r18',
        'cub_camion_r20_r22':'Cubiertas camion',
        'cub_agro_del':      'Cubiertas tractor chica',
        'cub_agro_tras_med': 'Cubiertas tractor mediana',
        'cub_agro_tras_gde': 'Cubiertas tractor grande',
        'cub_agro_tras_xgde':'Cubiertas tractor grande',
        'camara':            'Paquetes/cajas',
        'bateria_chica':     'Baterias chicas',
        'bateria_grande':    'Baterias camion',
        'lubricante_caja':   'Paquetes/cajas',
        'lubricante_tambor': 'Tanque aceite',
        'bulto_general':     'Paquetes/cajas',
    },
    'Franchi': {
        'cub_moto':          'Bultos',
        'cub_auto_r12_r14':  'Cub.auto',
        'cub_auto_r15_r19':  'Cub.camioneta',
        'cub_camion_r20_r22':'Cub.camion',
        'cub_agro_del':      'Bultos',
        'cub_agro_tras_med': 'Cub. agricola mediana',
        'cub_agro_tras_gde': 'Cub. agricola gde',
        'cub_agro_tras_xgde':'Cub. agricola gde',
        'camara':            'Bultos',
        'bateria_chica':     'Baterias chicas',
        'bateria_grande':    'Baterias grandes',
        'lubricante_caja':   'Bultos',
        'lubricante_tambor': None,
        'bulto_general':     'Bultos',
    },
}

print('Mapeo cargado para', len(MAPPING), 'agencias.')

Mapeo cargado para 14 agencias.


In [6]:
# ── Tabla comparativa por CATEGORÍA UNIFICADA ────────────────────────────
# Precio efectivo = precio_bruto × (1 - canje).
# GONFER: ajustado por IVA (+22%) y luego canje.
# Arzuaga: precios usados tal como están en la tabla (sin levante).
import unicodedata

IVA = 1.22

def norm(s):
    if s is None: return ''
    s = str(s).replace('\xa0', ' ')
    s = re.sub(r'\s+', ' ', s).strip().lower()
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

agency_prices_norm = {
    ag: {norm(k): v for k, v in prices.items()}
    for ag, prices in agency_prices.items()
}

def lookup_price(agency, unified_cat):
    cat_label = MAPPING.get(agency, {}).get(unified_cat)
    if cat_label is None:
        return None
    price = agency_prices_norm.get(agency, {}).get(norm(cat_label))
    if price and agency_meta.get(agency) == 'sin IVA':
        price = round(price * IVA, 2)
    return price

def lookup_effective(agency, unified_cat):
    raw = lookup_price(agency, unified_cat)
    if raw is None:
        return None
    return round(raw * (1 - agency_canje.get(agency, 0.0)), 2)

agencies_list = list(MAPPING.keys())

# Construir tabla
rows_comp = []
for ucat, ucat_name in UNIFIED_CATS.items():
    row = {'Categoría Unificada': ucat_name}
    effective_prices = {}
    for ag in agencies_list:
        raw = lookup_price(ag, ucat)
        eff = lookup_effective(ag, ucat)
        row[f'{ag}_bruto']    = raw
        row[f'{ag}_efectivo'] = eff
        if eff is not None:
            effective_prices[ag] = eff
    if effective_prices:
        best = min(effective_prices, key=effective_prices.get)
        row['MEJOR AGENCIA']          = best
        row['PRECIO EFECTIVO MÍNIMO'] = effective_prices[best]
    else:
        row['MEJOR AGENCIA'] = row['PRECIO EFECTIVO MÍNIMO'] = None
    rows_comp.append(row)

df_comp = pd.DataFrame(rows_comp).set_index('Categoría Unificada')

# Vista resumida: precio efectivo + frecuencia
eff_cols = [f'{ag}_efectivo' for ag in agencies_list]
df_view  = df_comp[['MEJOR AGENCIA', 'PRECIO EFECTIVO MÍNIMO'] + eff_cols].copy()
df_view.columns = ['MEJOR AGENCIA', 'PRECIO EF. MIN'] + agencies_list

# Fila de frecuencias al pie
freq_row_data = {ag: agency_freq[ag][0] for ag in agencies_list}
freq_row_data['MEJOR AGENCIA']   = '—'
freq_row_data['PRECIO EF. MIN'] = '—'
df_freq = pd.DataFrame([freq_row_data], index=['Frecuencia'])
df_view_con_freq = pd.concat([df_view, df_freq])

pd.set_option('display.float_format', '${:,.0f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 300)
print('=== COMPARATIVA PRECIO EFECTIVO (Mayo 2026, post-canje) ===')
print('Canje: NASAZZI 40% | MEGAM 25% | BULEVAR 25% | GONFER 15% | DAC 10%')
print('Arzuaga: precios sin levante (usados tal como están)')
print()
df_view_con_freq

=== COMPARATIVA PRECIO EFECTIVO (Mayo 2026, post-canje) ===
Canje: NASAZZI 40% | MEGAM 25% | BULEVAR 25% | GONFER 15% | DAC 10%
Arzuaga: precios sin levante (usados tal como están)



,MEJOR AGENCIA,PRECIO EF. MIN,DAC,NASAZZI,MEGAM,Transportes Nagar sas (Expreso Ruta 1),SELEGUIN,EXPRESO ROCHA,BULEVAR (ACC),PERICO,TRUJILLO,GONFER,3EME (El Chambon),Martin Escudero (Lascano),Arzuaga,Franchi
Cubierta Moto,BULEVAR (ACC),$73,$250,$118,$75,$109,$127,$152,$73,$87,$89,$118,$87,$122,$183,$99
Cubierta Auto R12-R14,MEGAM,$45,$267,$87,$45,$109,$114,$122,$73,$87,$89,$118,$87,$122,$88,$119
Cubierta Auto/Camioneta R15-R19,MEGAM,$68,$373,$118,$68,$147,$114,$171,$92,$101,$89,$179,$101,$146,$106,$148
Cubierta Camión R20-R22.5,MEGAM,$150,$401,$304,$150,$295,$266,$354,$183,$217,$198,$219,$217,$427,$232,$297
Cubierta Agrícola Delantera,Franchi,$99,"$1,337",$667,$150,$237,$127,$958,$175,$212,$178,$202,$212,$488,$464,$99
"Cubierta Agrícola Trasera Mediana 24-26""",MEGAM,$150,"$1,337",$667,$150,$653,$688,"$1,171",$412,$334,$441,$668,$334,"$1,098",$830,$693
"Cubierta Agrícola Trasera Grande 28-36""",MEGAM,$488,"$1,337","$1,102",$488,"$1,018","$1,111","$1,171",$824,$669,$658,"$1,232",$669,"$1,586","$1,000","$1,386"
"Cubierta Agrícola/Vial Extra Grande 38""+",MEGAM,$488,"$1,337","$2,394",$488,"$1,018","$1,316","$1,171","$1,006","$1,493",$658,"$1,232","$1,493","$1,586","$1,000","$1,386"
Cámara (cualquier tamaño),BULEVAR (ACC),$73,$332,$108,$75,$128,$127,$152,$73,$87,$89,$118,$87,$152,$146,$99
Batería Chica (≤110 Ah),MEGAM,$75,$171,$152,$75,$186,$139,$171,$110,$129,$84,$157,$129,$128,$183,$119


In [7]:
# ── TABLA DE MAPEO: que etiqueta usa cada agencia por tipo de producto ─────
#
# Esta tabla muestra de forma clara la unificacion:
# - Fila = agencia
# - Columna = categoria unificada
# - Celda = como LA LLAMA ESA AGENCIA en su tarifa + el precio bruto
#
# Si una celda parece incorrecta (la agencia usa otra categoria para ese producto),
# hay que corregir el MAPPING en la celda 'CATEGORIAS UNIFICADAS'.

rows = []
for ucat, ucat_name in UNIFIED_CATS.items():
    row = {'Agencia / Categoria unificada': ucat_name}
    for ag in agencies_list:
        label = MAPPING.get(ag, {}).get(ucat)
        if label is None:
            row[ag] = 'N/A'
            continue
        price = agency_prices_norm.get(ag, {}).get(norm(label))
        if price and agency_meta.get(ag) == 'sin IVA':
            price = round(price * IVA, 2)
        row[ag] = f'{label}  (${price:,.0f})' if price else label
    rows.append(row)

df_map = pd.DataFrame(rows).set_index('Agencia / Categoria unificada')

# Transponer: filas=agencias, columnas=categorias unificadas
df_T = df_map.T
df_T.index.name = 'Agencia'

pd.set_option('display.max_colwidth', 32)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 600)

print('COMO SE UNIFICA: cada celda muestra el nombre que usa la agencia + precio bruto')
print('Fila = agencia  |  Columna = categoria unificada')
print()
df_T


COMO SE UNIFICA: cada celda muestra el nombre que usa la agencia + precio bruto
Fila = agencia  |  Columna = categoria unificada



Agencia / Categoria unificada,Cubierta Moto,Cubierta Auto R12-R14,Cubierta Auto/Camioneta R15-R19,Cubierta Camión R20-R22.5,Cubierta Agrícola Delantera,"Cubierta Agrícola Trasera Mediana 24-26""","Cubierta Agrícola Trasera Grande 28-36""","Cubierta Agrícola/Vial Extra Grande 38""+",Cámara (cualquier tamaño),Batería Chica (≤110 Ah),Batería Grande (>110 Ah),Lubricante en caja/lata,Lubricante Tambor 205L,Bulto General
Agencia,,,,,,,,,,,,,,
DAC,Atado moto ($278),"Rodado 12,13,14 ($297)",Rodado 15 a 18 ($415),"Rodado 20 a 22,5 ($445)",Cubierta tractor AGRO 24 a 4...,Cubierta tractor AGRO 24 a 4...,Cubierta tractor AGRO 24 a 4...,Cubierta tractor AGRO 24 a 4...,Bulto hasta 15kg ($369),Bateria chica ($190),Bateria grande ($380),Bulto hasta 15kg ($369),"Tambor 205lts ($2,118)",Bulto hasta 15kg ($369)
NASAZZI,Rodado 15 a 19 ($196),Rodado 13 y 14 ($145),Rodado 15 a 19 ($196),"Rodado 20 a 22,5 ($506)","Rodado 24 y 25 ($1,111)","Rodado 24 y 25 ($1,111)","Rodado 26 a 32 ($1,836)",Cub viales de 38 a 46 extra ...,Bultos ($181),Bat hasta 110 amp ($254),Bat mayor 110 amp ($337),Bultos lubricantes ($254),"Tambor 205lts ($1,357)",Bultos ($181)
MEGAM,Bat y bultos ($100),Cub 13 y 14 ($60),Cubierta auto (15 a 17) ($90),Cubierta camion ($200),Tractor chico ($200),Tractor chico ($200),Tractor grande ($650),Tractor grande ($650),Bat y bultos ($100),Bat y bultos ($100),Bat y bultos ($100),Bat y bultos ($100),Tambor 205lts ($550),Bat y bultos ($100)
Transportes Nagar sas (Expreso Ruta 1),Auto ($109),Auto ($109),Camioneta ($147),Camion ($295),"Tractor del y 17,5 ($237)",Tractor traseras ($653),"Tractor grande ($1,018)","Tractor grande ($1,018)",Bultos ($128),Baterias ($186),Baterias ($186),Bultos ($128),Tambor ($717),Bultos ($128)
SELEGUIN,"Bultos, atados, bolsas, caja...",Cubierta auto ($115),Cubierta camioneta ($115),Cubierta camion ($268),"Bultos, atados, bolsas, caja...",Tractor mediana 24 a 26 ($695),"Tractor grande 28 a 30 ($1,...",Tractor extra grande +30 ($...,"Bultos, atados, bolsas, caja...",Baterias chicas ($140),Baterias grandes ($220),"Bultos, atados, bolsas, caja...",N/A,"Bultos, atados, bolsas, caja..."
EXPRESO ROCHA,Bultos camaras ($152),Cubierta chica hasta 205/70....,Cubierta grande hasta 750.16...,Cubierta camion ($354),Neumatico agricolas y viales...,Neum. Agricolas y viales mas...,Neum. Agricolas y viales mas...,Neum. Agricolas y viales mas...,Bultos camaras ($152),Baterias ($171),Baterias ($171),Bultos camaras ($152),N/A,Bultos camaras ($152)
BULEVAR (ACC),Atado cub moto ($98),Cub auto hasta rod 14 ($98),Cub camioneta 15 a 19 ($122),Cub camion ($244),Cub tractor chica (hasta 22)...,Cub tractor mediana (24) ($...,Cub tractor grande (de 25 a ...,"Cub vial rod 38 a 42 ($1,342)",Bolsas ($98),Bat chicas ($146),Bat grandes ($220),Caja aceite 1 y 4lts ($152),Tambor 205lts ($915),Bolsas ($98)
PERICO,Atado cub moto ($87),Cub auto hasta rod 14 ($87),Cub camioneta 15 a 19 ($101),Cub camion ($217),Cub tractor chica ($212),Cub tractor mediana ($334),Cub tractor grande ($669),"Cub vial rod 38 a 42 ($1,493)",Bolsas ($87),Bat chicas ($129),Bat grandes ($167),Caja aceite 1 y 4lts ($115),Tambor 205lts ($830),Bolsas ($87)
TRUJILLO,Bultos ($90),Paseo hasta 16 ($90),Paseo hasta 16 ($90),Camion ($200),Agricola delantera ($180),Agricola mediana ($445),Agricola extra grande ($665),Agricola extra grande ($665),Bultos ($90),Bat hasta 150 amp ($85),Bat + 150 amp ($125),Cajas lubricantes ($100),Tambor 205lts ($600),Bultos ($90)


In [8]:
# ── Clasificar productos de Libro1 ───────────────────────────────────────
# Los tamaños están en las descripciones en formato estándar (no necesita web scraping):
#   - Cubiertas: "7.50 - 16", "14.9-24", "175/65 TR 14", "R 22.5", "KR16"
#   - Baterías: "90A", "120A"
#   - Lubricantes: "1LT", "4LT", "20LT", "205L"
# El regex de 3 pasos cubre 100% de las familias críticas.

def extract_rim(desc):
    d = str(desc).upper()
    # Paso 1: guión con espacios ('6.50 - 16', '14.9 - 24')
    m = re.search(r'\d\s+-\s+(\d{2,3})\b', d)
    if m: return float(m.group(1))
    # Paso 2: guión sin espacios ('7.50-16', '11L-16', '14.9-24')
    m = re.search(r'-(\d{2,3})\b', d)
    if m: return float(m.group(1))
    # Paso 3: notación R ('R15', 'TR 14', 'KR16', 'R 22.5')
    m = re.search(r'R\s*(\d{2}(?:\.\d)?)\b', d)
    if m: return float(m.group(1))
    return None

def extract_amp(desc):
    m = re.search(r'\b(\d{2,3})A\b', str(desc).upper())
    return int(m.group(1)) if m else None

def classify(familia, desc):
    f = str(familia).upper()
    d = str(desc).upper()

    if 'LUBRICANTE' in f:
        return 'lubricante_tambor' if ('TAMBOR' in d or '205' in d) else 'lubricante_caja'

    if 'BATERIA' in f:
        amp = extract_amp(desc)
        if amp: return 'bateria_grande' if amp > 110 else 'bateria_chica'
        return 'bateria_chica'

    # Cámaras y accesorios → siempre "camara" (no importa el rodado)
    if 'ACCES' in f or 'CAM MOTO' in f:
        return 'camara'

    if 'MOTO' in f and 'CAM' not in f:
        return 'cub_moto'

    # Gigante/OTR (camiones/buses): siempre R17.5-R22.5
    if 'GIGANTE' in f:
        return 'cub_camion_r20_r22'

    if 'VIAL' in f or 'IND.' in f:
        rim = extract_rim(desc)
        return 'cub_agro_tras_xgde' if (rim and rim >= 38) else 'cub_agro_tras_gde'

    if 'AGR DEL' in f or 'AGRICOLA DEL' in f:
        rim = extract_rim(desc)
        if rim:
            if rim >= 38: return 'cub_agro_tras_xgde'
            if rim >= 28: return 'cub_agro_tras_gde'
            if rim >= 24: return 'cub_agro_tras_med'
        return 'cub_agro_del'

    if 'AGR TRAS' in f or 'AGRICOLA TRAS' in f:
        rim = extract_rim(desc)
        if rim:
            if rim >= 38: return 'cub_agro_tras_xgde'
            if rim >= 30: return 'cub_agro_tras_gde'
            if rim >= 24: return 'cub_agro_tras_med'
        return 'cub_agro_tras_med'

    # PASEO: autos de paseo — se limita a r12_r14 o r15_r19 (no llegan a "camión")
    if 'PASEO' in f:
        rim = extract_rim(desc)
        if rim and rim <= 14: return 'cub_auto_r12_r14'
        return 'cub_auto_r15_r19'

    # Camioneta / Pick up: pueden ser R15-R22
    if 'CAMIONETA' in f or 'PICK UP' in f:
        rim = extract_rim(desc)
        if rim:
            if rim <= 14: return 'cub_auto_r12_r14'
            if rim <= 19: return 'cub_auto_r15_r19'
            return 'cub_camion_r20_r22'
        return 'cub_auto_r15_r19'

    return 'bulto_general'

df['categoria_unif']   = df.apply(lambda r: classify(r['Familia'], r['Descripción de Producto']), axis=1)
df['categoria_nombre'] = df['categoria_unif'].map(UNIFIED_CATS)

resumen = df.groupby('categoria_nombre').size().sort_values(ascending=False)
print(f'Total productos: {len(df)}')
print()
print(resumen.to_string())

Total productos: 1437

categoria_nombre
Cubierta Auto/Camioneta R15-R19             588
Cubierta Moto                               178
Cubierta Camión R20-R22.5                   147
Cubierta Agrícola Trasera Grande 28-36"     139
Cubierta Agrícola Trasera Mediana 24-26"     86
Cámara (cualquier tamaño)                    86
Cubierta Agrícola/Vial Extra Grande 38"+     60
Cubierta Auto R12-R14                        48
Cubierta Agrícola Delantera                  41
Lubricante en caja/lata                      26
Batería Grande (>110 Ah)                     16
Batería Chica (≤110 Ah)                      13
Lubricante Tambor 205L                        9


## Cómo funciona la clasificación

Cada producto pasa por **dos pasos**:

### Paso 1 — Familia del producto

```
Familia
│
├── LUBRICANTES        → ¿"TAMBOR"/"205" en descripción? → lubricante_tambor
│                                                         → lubricante_caja
├── BATERIAS           → extraer amperaje ("90A")
│                         ≤110A → bateria_chica  |  >110A → bateria_grande
├── ACCES./CAM MOTO    → camara  (sin importar el rodado)
├── MOTO               → cub_moto
├── GIGANTE/OTR        → cub_camion_r20_r22
├── VIALES/IND.        → rodado ≥38" → xgde  |  resto → gde
├── AGR DEL.           → rodado ≥38"→xgde | ≥28"→gde | ≥24"→med | resto→del
├── AGR TRAS.          → rodado ≥38"→xgde | ≥30"→gde | ≥24"→med
├── PASEO              → rodado ≤14" → r12_r14  |  resto → r15_r19
└── CAMIONETA/PICK UP  → rodado ≤14"→r12 | ≤19"→r15 | ≥20"→camion
```

### Paso 2 — Extraer rodado de la descripción (3 intentos)

```
"14.9 - 24  14PR TT"  → intento 1: dígito + espacios + guión + 2-3 dígitos  → 24"
"7.50-16    8PR TT"   → intento 2: guión sin espacio + 2-3 dígitos          → 16"
"175/65 TR 14"        → intento 3: R + espacio opcional + 2 dígitos         → 14"
"11L-16     8PR TT"   → intento 2: -16                                      → 16"
```

In [9]:
# ── Verificación: 3 ejemplos reales por familia con clasificación paso a paso ──
desc_col = 'Descripción de Producto'

print(f'{"Descripción (truncada)":<55} {"Rodado":>7}  {"Categoría unificada"}')
print('─' * 110)

fam_order = [
    'PASEO', 'PASEO OTRAS MAR', 'PICK UP', 'CAMIONETA OTRAS',
    'GIGANTE', 'GIGANTE OTRAS M',
    'AGR DEL. OTRAS', 'AGRICOLA DEL.',
    'AGR TRAS. OTRAS', 'AGRICOLA TRAS.',
    'VIALES OTRAS M', 'IND. Y VIALES',
    'BATERIAS', 'ACCES. OTRAS M', 'CAM MOTO OTRAS',
    'MOTO', 'MOTO OTRAS M.',
    'LUBRICANTES',
]

for fam in fam_order:
    sub = df[df['Familia'] == fam]
    if sub.empty:
        continue
    print(f'\n  [ Familia: {fam} ]')
    for _, row in sub.head(3).iterrows():
        d    = str(row[desc_col])
        rim  = extract_rim(d)
        amp  = extract_amp(d)
        cat  = classify(row['Familia'], d)
        if 'BATERIA' in fam.upper():
            dim_str = f'{amp}A' if amp else '—'
        else:
            dim_str = f'{rim}"' if rim else '—'
        print(f'  {d[:55]:<55} {dim_str:>7}  → {UNIFIED_CATS.get(cat, cat)}')

Descripción (truncada)                                   Rodado  Categoría unificada
──────────────────────────────────────────────────────────────────────────────────────────────────────────────

  [ Familia: PASEO ]
  PIRELLI P1 CINT       175/65 TR 14                        14.0"  → Cubierta Auto R12-R14
  PIRELLI P7 CINT       205/60 HR 15                        15.0"  → Cubierta Auto/Camioneta R15-R19
  PIRELLI P7 CINT       205/55 VR 16 (KS)                   16.0"  → Cubierta Auto/Camioneta R15-R19

  [ Familia: PASEO OTRAS MAR ]
  MOMO M20 PRO VT   185/65 R 15 88H                         15.0"  → Cubierta Auto/Camioneta R15-R19
  MOMO M20 PRO VT      185/60 R 15  88H XL                  15.0"  → Cubierta Auto/Camioneta R15-R19
  MOMO TOPRUN M300 VT   205/55 R 16 94V XL                  16.0"  → Cubierta Auto/Camioneta R15-R19

  [ Familia: PICK UP ]
  PIRELLI SCORPION ATR XL 175/70 HR 14                      14.0"  → Cubierta Auto R12-R14
  PIRELLI SCORPION XL   225/60 HR 18 10


  [ Familia: AGR TRAS. OTRAS ]
  GTK AS100   14.9 - 24  14PR TT R1 EX                      24.0"  → Cubierta Agrícola Trasera Mediana 24-26"
  GTK AS100   18.4 - 34  14PR TT R1 EX                      34.0"  → Cubierta Agrícola Trasera Grande 28-36"
  GTK RS200   520/85 R 42 (20.8 R42) 157/157A8 TL R1W EX    42.0"  → Cubierta Agrícola/Vial Extra Grande 38"+

  [ Familia: AGRICOLA TRAS. ]
  PIRELLI TM95   23.1 - 30 12PR R1 TT EX                    30.0"  → Cubierta Agrícola Trasera Grande 28-36"
  PIRELLI SX02 IF380/90 R 46 168D  R1 TL EX                 46.0"  → Cubierta Agrícola/Vial Extra Grande 38"+
  PIRELLI PD22   14.9 - 24  8PR R2 TT EX                    24.0"  → Cubierta Agrícola Trasera Mediana 24-26"

  [ Familia: VIALES OTRAS M ]
  GTK CK50      6.00 - 9  12PR TT ELEVADOR                      —  → Cubierta Agrícola Trasera Grande 28-36"
  GTK LD90      19.5L - 24 (500/70-24)  14PR TL RETRO TRA   24.0"  → Cubierta Agrícola Trasera Grande 28-36"
  GTK CK50      7.00 - 12  14P


  [ Familia: BATERIAS ]
  ENERGIZER PLUS   50G FREE D 90A                             90A  → Batería Chica (≤110 Ah)
  ENERGIZER PLUS   36J FREE D 65A                             65A  → Batería Chica (≤110 Ah)
  ENERGIZER PLUS   60H FREE D 110A                           110A  → Batería Chica (≤110 Ah)

  [ Familia: ACCES. OTRAS M ]
  CAMARA SELETTO KR16          TR15                         16.0"  → Cámara (cualquier tamaño)
  CAMARA SELETTO   14.9/13  -24   TR218A EX                 24.0"  → Cámara (cualquier tamaño)
  CAMARA SELETTO       7.50 - 16 TR15 EX                    16.0"  → Cámara (cualquier tamaño)

  [ Familia: CAM MOTO OTRAS ]
  CAMARA SELETTO 2.75/3.00 - 21 TR4                         21.0"  → Cámara (cualquier tamaño)
  CAMARA SELETTO 3.00/3.25 - 18 TR4                         18.0"  → Cámara (cualquier tamaño)
  CAMARA SELETTO 80/100    - 14 TR4                         14.0"  → Cámara (cualquier tamaño)

  [ Familia: MOTO ]
  PIRELLI SUPER CITY       90/90-18 TL/TT T

In [10]:
# ── Precio de envío por producto × agencia (bruto y efectivo) ─────────────
for ag in agencies_list:
    df[f'{ag}_bruto']    = df['categoria_unif'].apply(lambda c: lookup_price(ag, c))
    df[f'{ag}_efectivo'] = df['categoria_unif'].apply(lambda c: lookup_effective(ag, c))

eff_cols = [f'{ag}_efectivo' for ag in agencies_list]
df['PRECIO_EF_MIN']  = df[eff_cols].min(axis=1)
df['MEJOR_AGENCIA']  = df[eff_cols].idxmin(axis=1).str.replace('_efectivo', '', regex=False)

out_cols = (['Prod.', 'Descripción de Producto', 'Familia', 'categoria_nombre',
             'MEJOR_AGENCIA', 'PRECIO_EF_MIN'] + eff_cols)
df_out = df[out_cols].copy()
df_out.columns = (['Prod.', 'Descripción', 'Familia', 'Categoría',
                   'MEJOR AGENCIA', 'PRECIO EF. MIN'] + agencies_list)

print(f'Productos clasificados: {len(df_out)}')
print('\nMejor agencia por cantidad de productos (precio efectivo):')
print(df_out['MEJOR AGENCIA'].value_counts().to_string())
df_out.head(20)

Productos clasificados: 1437

Mejor agencia por cantidad de productos (precio efectivo):
MEJOR AGENCIA
MEGAM            1132
BULEVAR (ACC)     264
Franchi            41


,Prod.,Descripción,Familia,Categoría,MEJOR AGENCIA,PRECIO EF. MIN,DAC,NASAZZI,MEGAM,Transportes Nagar sas (Expreso Ruta 1),SELEGUIN,EXPRESO ROCHA,BULEVAR (ACC),PERICO,TRUJILLO,GONFER,3EME (El Chambon),Martin Escudero (Lascano),Arzuaga,Franchi
0,"$20,046",CAMARA SELETTO KR16 ...,ACCES. OTRAS M,Cámara (cualquier tamaño),BULEVAR (ACC),$73,$332,$108,$75,$128,$127,$152,$73,$87,$89,$118,$87,$152,$146,$99
1,"$20,132",CAMARA SELETTO 14.9/13 -2...,ACCES. OTRAS M,Cámara (cualquier tamaño),BULEVAR (ACC),$73,$332,$108,$75,$128,$127,$152,$73,$87,$89,$118,$87,$152,$146,$99
2,"$20,052",CAMARA SELETTO 7.50 - ...,ACCES. OTRAS M,Cámara (cualquier tamaño),BULEVAR (ACC),$73,$332,$108,$75,$128,$127,$152,$73,$87,$89,$118,$87,$152,$146,$99
3,"$20,085",CAMARA SELETTO 18.4 -3...,ACCES. OTRAS M,Cámara (cualquier tamaño),BULEVAR (ACC),$73,$332,$108,$75,$128,$127,$152,$73,$87,$89,$118,$87,$152,$146,$99
4,"$20,040",CAMARA SELETTO KR14 ...,ACCES. OTRAS M,Cámara (cualquier tamaño),BULEVAR (ACC),$73,$332,$108,$75,$128,$127,$152,$73,$87,$89,$118,$87,$152,$146,$99
5,"$20,058",CAMARA SELETTO 12-1...,ACCES. OTRAS M,Cámara (cualquier tamaño),BULEVAR (ACC),$73,$332,$108,$75,$128,$127,$152,$73,$87,$89,$118,$87,$152,$146,$99
6,"$20,048",CAMARA SELETTO 6.50/7.00-16 ...,ACCES. OTRAS M,Cámara (cualquier tamaño),BULEVAR (ACC),$73,$332,$108,$75,$128,$127,$152,$73,$87,$89,$118,$87,$152,$146,$99
7,"$20,049",CAMARA SELETTO 7.50 R 16 ...,ACCES. OTRAS M,Cámara (cualquier tamaño),BULEVAR (ACC),$73,$332,$108,$75,$128,$127,$152,$73,$87,$89,$118,$87,$152,$146,$99
8,"$20,083",CAMARA SELETTO 11.2/12.4-2...,ACCES. OTRAS M,Cámara (cualquier tamaño),BULEVAR (ACC),$73,$332,$108,$75,$128,$127,$152,$73,$87,$89,$118,$87,$152,$146,$99
9,"$20,044",CAMARA SELETTO KR15 ...,ACCES. OTRAS M,Cámara (cualquier tamaño),BULEVAR (ACC),$73,$332,$108,$75,$128,$127,$152,$73,$87,$89,$118,$87,$152,$146,$99


In [11]:
# ── Ranking por categoría: precio efectivo + canje + frecuencia ──────────
print('=== RANKING POR CATEGORÍA  (precio efectivo · canje · frecuencia) ===\n')
for ucat, ucat_name in UNIFIED_CATS.items():
    scores = {}
    for ag in agencies_list:
        bruto = lookup_price(ag, ucat)
        eff   = lookup_effective(ag, ucat)
        if eff is not None:
            freq_label, freq_days = agency_freq[ag]
            scores[ag] = (bruto, eff, freq_label, freq_days)
    if not scores:
        continue
    sorted_scores = sorted(scores.items(), key=lambda x: x[1][1])
    print(f'{ucat_name}:')
    for i, (ag, (bruto, eff, flabel, fdays)) in enumerate(sorted_scores, 1):
        canje_pct = agency_canje.get(ag, 0) * 100
        canje_str = f'canje {canje_pct:.0f}%' if canje_pct > 0 else 'sin canje'
        freq_str  = f'{flabel} ({fdays}d/sem)'
        note      = '  <- MEJOR' if i == 1 else ''
        print(f'  {i:2}. {ag:<45}  ${bruto:>8,.0f} -> ${eff:>8,.0f}  {canje_str:<12}  {freq_str}{note}')
    print()

=== RANKING POR CATEGORÍA  (precio efectivo · canje · frecuencia) ===

Cubierta Moto:
   1. BULEVAR (ACC)                                  $      98 -> $      73  canje 25%     L a V (5d/sem)  <- MEJOR
   2. MEGAM                                          $     100 -> $      75  canje 25%     L a V (5d/sem)
   3. PERICO                                         $      87 -> $      87  sin canje     Mar y Jue (2d/sem)
   4. 3EME (El Chambon)                              $      87 -> $      87  sin canje     Bajo pedido (1d/sem)
   5. TRUJILLO                                       $      90 -> $      89  canje 1%      L a V (5d/sem)
   6. Franchi                                        $     100 -> $      99  canje 1%      L a V (5d/sem)
   7. Transportes Nagar sas (Expreso Ruta 1)         $     109 -> $     109  sin canje     L a V (5d/sem)
   8. NASAZZI                                        $     196 -> $     118  canje 40%     L a V (5d/sem)
   9. GONFER                                  

In [12]:
# ── OPTIMIZADOR DE ENVÍOS ─────────────────────────────────────────────────
# Dado un pedido {codigo_producto: cantidad}, calcula:
#   MODO 1 - Misma agencia:   ranking de agencias con el costo total del pedido
#   MODO 2 - Por producto:    cada producto va a la agencia más barata para ese tipo
# La diferencia entre ambos totales muestra el costo de consolidar en una sola agencia.

def buscar_producto(codigo):
    """Busca el producto en df por código; tolera int, float y string."""
    cod_str = str(codigo).strip().rstrip('.0') if str(codigo).endswith('.0') else str(codigo).strip()
    # Intentar match exacto sobre la columna Prod. como string
    mask = df['Prod.'].astype(str).str.strip().str.rstrip('0').str.rstrip('.') == cod_str
    hit = df[mask]
    if hit.empty:
        # Segundo intento: comparación numérica
        try:
            mask2 = df['Prod.'] == float(codigo)
            hit = df[mask2]
        except (ValueError, TypeError):
            pass
    return hit.iloc[0] if not hit.empty else None


def calcular_envio(pedido, mostrar_detalle=True):
    """
    pedido: dict {codigo_producto: cantidad}
    Ej: {20046: 10, 31186: 5, 130108: 20}
    """
    # ── Resolver productos ────────────────────────────────────────────────
    lineas = []
    for cod, qty in pedido.items():
        prod = buscar_producto(cod)
        if prod is None:
            print(f"  ADVERTENCIA: codigo {cod!r} no encontrado en Libro1")
            continue
        ucat = prod['categoria_unif']
        lineas.append({
            'codigo':      cod,
            'descripcion': str(prod['Descripcion de Producto'] if 'Descripcion de Producto' in prod.index
                               else prod['Descripción de Producto'])[:55],
            'familia':     prod['Familia'],
            'categoria':   prod['categoria_nombre'],
            'ucat':        ucat,
            'cantidad':    qty,
        })

    if not lineas:
        print("No se encontraron productos en el pedido.")
        return None, None

    if mostrar_detalle:
        print("Productos del pedido:")
        for l in lineas:
            print(f"  {l['codigo']}  {l['descripcion'][:50]:<50}  qty={l['cantidad']:>4}  → {l['categoria']}")
        print()

    # ── MODO 1: Toda la carga en UNA sola agencia ─────────────────────────
    modo1 = []
    for ag in agencies_list:
        freq_label, freq_days = agency_freq[ag]
        total = 0.0
        sin_precio = []
        detalles = []
        for l in lineas:
            p = lookup_effective(ag, l['ucat'])
            if p is None:
                sin_precio.append(l['descripcion'][:30])
            else:
                subtotal = p * l['cantidad']
                total += subtotal
                detalles.append((l['descripcion'], l['cantidad'], p, subtotal))
        modo1.append({
            'Agencia':       ag,
            'Total (ef.)':   round(total, 0) if not sin_precio else None,
            'Frecuencia':    freq_label,
            'Dias/sem':      freq_days,
            'Sin precio':    ', '.join(sin_precio) if sin_precio else '',
            '_detalles':     detalles,
        })

    df_m1 = (pd.DataFrame(modo1)
               .drop(columns=['_detalles'])
               .sort_values('Total (ef.)'))

    # ── MODO 2: Cada producto a su MEJOR agencia ──────────────────────────
    total_m2 = 0.0
    modo2 = []
    for l in lineas:
        precios_ag = {ag: lookup_effective(ag, l['ucat'])
                      for ag in agencies_list
                      if lookup_effective(ag, l['ucat']) is not None}
        if not precios_ag:
            modo2.append({
                'Codigo':       l['codigo'],
                'Descripcion':  l['descripcion'],
                'Categoria':    l['categoria'],
                'Cantidad':     l['cantidad'],
                'Mejor Agencia':'SIN PRECIO',
                'Precio ef.':   None,
                'Subtotal':     None,
                'Frecuencia':   '',
            })
            continue
        mejor_ag = min(precios_ag, key=precios_ag.get)
        precio   = precios_ag[mejor_ag]
        subtotal = precio * l['cantidad']
        total_m2 += subtotal
        modo2.append({
            'Codigo':       l['codigo'],
            'Descripcion':  l['descripcion'],
            'Categoria':    l['categoria'],
            'Cantidad':     l['cantidad'],
            'Mejor Agencia':mejor_ag,
            'Precio ef.':   precio,
            'Subtotal':     round(subtotal, 0),
            'Frecuencia':   agency_freq[mejor_ag][0],
        })

    df_m2 = pd.DataFrame(modo2)

    # ── Resumen de agencias necesarias en modo 2 ──────────────────────────
    agencias_m2 = df_m2['Mejor Agencia'].value_counts()

    return df_m1, df_m2, round(total_m2, 0), agencias_m2


print("Funcion calcular_envio() lista.")
print("Uso:  df_m1, df_m2, total_m2, agencias_m2 = calcular_envio(PEDIDO)")


Funcion calcular_envio() lista.
Uso:  df_m1, df_m2, total_m2, agencias_m2 = calcular_envio(PEDIDO)


In [13]:
# ── PEDIDO — editar aqui con los productos y cantidades ──────────────────
#
# Formato: {codigo_producto: cantidad}
# El codigo es el valor de la columna "Prod." de Libro1.
# Ejemplos de codigos: 20046 (camara auto), 31186 (bateria), 130108 (aceite 1L)

PEDIDO = {
    20046:  10,   # CAMARA SELETTO KR16 TR15
    20052:   5,   # CAMARA SELETTO 7.50-16
    31186:   8,   # BATERIA ENERGIZER 90A (chica)
    31189:   3,   # BATERIA ENERGIZER 120A (grande)
    130108: 20,   # ENI I-RIDE 20W-50 1LT
}

# ── Calcular ──────────────────────────────────────────────────────────────
df_m1, df_m2, total_m2, agencias_m2 = calcular_envio(PEDIDO)

pd.set_option('display.float_format', '${:,.0f}'.format)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', 30)

# ── MODO 1: Una sola agencia ──────────────────────────────────────────────
print("=" * 70)
print("MODO 1: TODA LA CARGA EN UNA SOLA AGENCIA")
print("=" * 70)
print(df_m1[df_m1['Sin precio'] == ''].to_string(index=False))
if df_m1['Sin precio'].any():
    print()
    print("Agencias que no cubren todos los productos del pedido:")
    print(df_m1[df_m1['Sin precio'] != ''][['Agencia','Sin precio']].to_string(index=False))

# ── MODO 2: Cada producto a la mejor agencia ──────────────────────────────
print()
print("=" * 70)
print("MODO 2: CADA PRODUCTO A SU MEJOR AGENCIA")
print("=" * 70)
print(df_m2.to_string(index=False))
print()
print(f"Total Modo 2: ${total_m2:,.0f}")
print()
print("Agencias necesarias en Modo 2:")
for ag, n in agencias_m2.items():
    freq_label, freq_days = agency_freq.get(ag, ('?', 0))
    print(f"  {ag:<45}  {n} producto(s)  |  {freq_label} ({freq_days}d/sem)")

# ── Comparativa final ─────────────────────────────────────────────────────
mejor_m1_total = df_m1[df_m1['Sin precio'] == '']['Total (ef.)'].min()
mejor_m1_ag    = df_m1[df_m1['Sin precio'] == ''].iloc[0]['Agencia']

print()
print("=" * 70)
print("COMPARATIVA")
print("=" * 70)
print(f"  Modo 1 (mejor agencia unica):   ${mejor_m1_total:>10,.0f}  [{mejor_m1_ag}]")
print(f"  Modo 2 (split por producto):    ${total_m2:>10,.0f}  ({len(agencias_m2)} agencia(s))")
diferencia = mejor_m1_total - total_m2
if diferencia > 0:
    print(f"  Costo de consolidar:            ${diferencia:>10,.0f}  mas caro en Modo 1")
    print(f"  -> Conviene dividir el envio entre {len(agencias_m2)} agencia(s)")
else:
    print(f"  Ahorro de consolidar:           ${-diferencia:>10,.0f}  mas barato en Modo 1")
    print(f"  -> Conviene mandar todo con {mejor_m1_ag}")


Productos del pedido:
  20046  CAMARA SELETTO KR16          TR15                   qty=  10  → Cámara (cualquier tamaño)
  20052  CAMARA SELETTO       7.50 - 16 TR15 EX              qty=   5  → Cámara (cualquier tamaño)
  31186  ENERGIZER PLUS   50G FREE D 90A                     qty=   8  → Batería Chica (≤110 Ah)
  31189  ENERGIZER PLUS   70N FREE D 120A                    qty=   3  → Batería Grande (>110 Ah)
  130108  ENI I-RIDE SPECIAL 20W-50 1LT BI                    qty=  20  → Lubricante en caja/lata

MODO 1: TODA LA CARGA EN UNA SOLA AGENCIA
                               Agencia  Total (ef.)  Frecuencia  Dias/sem Sin precio
                                 MEGAM       $3,450       L a V         5           
                              TRUJILLO       $4,361       L a V         5           
                         BULEVAR (ACC)       $4,758       L a V         5           
                               Franchi       $4,772       L a V         5           
                   

In [14]:
# ── Exportar a Excel ─────────────────────────────────────────────────────
output_file = 'comparativa_resultado.xlsx'

df_agencias = pd.DataFrame([
    {'Agencia': ag,
     'Canje %': round(agency_canje[ag]*100, 0),
     'Factor efectivo': round(1 - agency_canje[ag], 4),
     'IVA': agency_meta[ag],
     'Frecuencia': agency_freq[ag][0],
     'Dias/semana': agency_freq[ag][1]}
    for ag in agencies_list
]).sort_values('Canje %', ascending=False)

cols_exp = (['Prod.', 'Descripción de Producto', 'Familia', 'categoria_nombre',
             'MEJOR_AGENCIA', 'PRECIO_EF_MIN']
            + [f'{ag}_efectivo' for ag in agencies_list]
            + [f'{ag}_bruto'    for ag in agencies_list])
df_exp_prod = df[cols_exp].copy()
df_exp_prod.columns = (['Codigo', 'Descripcion', 'Familia', 'Categoria Unificada',
                         'Mejor Agencia', 'Precio Ef. Min']
                        + [f'{ag} (ef.)'    for ag in agencies_list]
                        + [f'{ag} (bruto)'  for ag in agencies_list])

df_fam = (df[['Familia','categoria_nombre','MEJOR_AGENCIA','PRECIO_EF_MIN']]
          .groupby(['Familia','categoria_nombre','MEJOR_AGENCIA'])['PRECIO_EF_MIN']
          .mean().reset_index())
df_fam.columns = ['Familia','Categoria Unificada','Mejor Agencia','Precio Ef. Promedio']

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_agencias.to_excel(writer,    sheet_name='Agencias', index=False)
    df_view_con_freq.to_excel(writer, sheet_name='Por Categoria (ef.)')
    df_exp_prod.to_excel(writer,    sheet_name='Por Producto', index=False)
    df_fam.to_excel(writer,         sheet_name='Por Familia', index=False)

print(f'Exportado: {output_file}')
print('  Hoja 1 Agencias:           canje % + frecuencia')
print('  Hoja 2 Por Categoria:      precios efectivos + fila frecuencias')
print('  Hoja 3 Por Producto:       1437 productos con precio ef. y bruto')
print('  Hoja 4 Por Familia:        resumen por familia')

Exportado: comparativa_resultado.xlsx
  Hoja 1 Agencias:           canje % + frecuencia
  Hoja 2 Por Categoria:      precios efectivos + fila frecuencias
  Hoja 3 Por Producto:       1437 productos con precio ef. y bruto
  Hoja 4 Por Familia:        resumen por familia
